# 03 Relation Extraction With Qwen Hybrid

Build graph/relation artifacts in dry-run mode. No Neo4j or Supabase writes happen in Kaggle.

In [ ]:
import os, subprocess, sys
from pathlib import Path

PROJECT_DIR = Path(os.environ.get('PROJECT_DIR', '/kaggle/input/tuvi-battu-graphrag/tuvi-battu-graphrag'))
if not PROJECT_DIR.exists():
    PROJECT_DIR = Path.cwd()
OUTPUT_ROOT = Path('/kaggle/working/w3_local_outputs')
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
REPORTS = OUTPUT_ROOT / 'reports'
STATE = OUTPUT_ROOT / 'state'
STRATEGIES = ['chunk_fixed_512', 'chunk_structure_parent_child', 'chunk_semantic_embedding_bge_m3']


In [ ]:
for strategy in STRATEGIES:
    cmd = [
        sys.executable, '-B', str(PROJECT_DIR / 'scripts' / 'write_graph_provenance.py'),
        '--chunks-input', str(OUTPUT_ROOT / 'chunks' / strategy),
        '--entities-input', str(OUTPUT_ROOT / 'entities' / strategy / f'{strategy}_entities.jsonl'),
        '--chunking-strategy', strategy,
        '--relation-mode', 'hybrid',
        '--dry-run',
        '--summary-output', str(REPORTS / f'{strategy}_graph_write_summary.json'),
        '--relation-review-output', str(REPORTS / f'{strategy}_relation_review.json'),
        '--state-output', str(STATE / f'{strategy}_graph_relation_state.json'),
        '--resume',
        '--llm-backend', 'local',
        '--model', 'Qwen/Qwen2.5-7B-Instruct',
        '--local-llm-quantization', '4bit',
        '--local-llm-max-new-tokens', '1024',
    ]
    subprocess.run(cmd, cwd=PROJECT_DIR, check=True)
